# Hydrogen diffusion in Pd: from CP-PAW protocols to a diffusion constant

This notebook walks through the post-processing chain on the committed calculations: read the eleven constrained relaxations, build the energy profile, fit the barrier and the two wells, and evaluate jump rates and the diffusion constant with harmonic transition-state theory.

In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pdhdiff.constants import A_PD_ANG, D_EXP_298K_M2_S, HARTREE_EV, MASS_H_AMU
from pdhdiff.cppaw_io import parse_prot
from pdhdiff.profile import collect_path, fit_octahedral_well, fit_tetrahedral_well, locate_barrier
from pdhdiff.structure import hop_geometry, path_length_ang
from pdhdiff.tst import SiteModel, arrhenius_fit, attempt_frequency, diffusion_constant, force_constant_si, vibrational_quantum_mev
from pdhdiff.plotting import plot_energy_profile, plot_arrhenius

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
CALC = ROOT / "calculations"

## 1. The two interstitial sites

Unconstrained relaxations of one H atom in the 32-atom supercell.

In [2]:
octa = parse_prot(CALC / "03_h_octahedral" / "stage2_relaxation" / "pd_super_octa.prot.gz")
tetra = parse_prot(CALC / "03_h_tetrahedral" / "stage2_relaxation" / "pd_super_tetra.prot.gz")
for name, prot in (("octahedral", octa), ("tetrahedral", tetra)):
    h = prot.atom("H_01")
    print(f"{name:12s} E = {prot.final_energy:.7f} H  k-points = {prot.n_kpoints}  H at {h.position.round(3)} A  steps = {prot.n_iterations}")
print(f"E_T - E_O = {(tetra.final_energy - octa.final_energy) * HARTREE_EV * 1e3:.1f} meV")

octahedral   E = -960.8412761 H  k-points = 8  H at [0.006 0.002 1.945] A  steps = 337
tetrahedral  E = -960.8381902 H  k-points = 8  H at [0.973 0.972 2.917] A  steps = 378
E_T - E_O = 84.0 meV


## 2. The energy profile

`collect_path` reads every `g_*` directory, takes the final energy of the stage-2 relaxation and cross-checks the reaction coordinate three ways: from the input H position, from the constraint value CP-PAW printed, and from the relaxed Cartesian positions.

In [3]:
points = collect_path(CALC / "04_path_octa_to_tetra")
g = np.array([p.g_nominal for p in points])
e = np.array([(p.energy_h - points[0].energy_h) * HARTREE_EV for p in points])
print("   g   g(constraint)  g(relaxed)   E-E_O [meV]  max Pd disp [A]")
for p, ee in zip(points, e):
    print(f"{p.g_nominal:4.1f}   {p.g_constraint:10.4f}   {p.g_relaxed:9.4f}   {ee*1e3:10.1f}   {p.max_pd_displacement_ang:8.3f}")

   g   g(constraint)  g(relaxed)   E-E_O [meV]  max Pd disp [A]
 0.0       0.0001      0.0000          0.0      0.016
 0.1       0.0994      0.1000          5.5      0.021
 0.2       0.2001      0.2000         22.8      0.031
 0.3       0.2994      0.3000         61.6      0.051
 0.4       0.4000      0.4000        114.4      0.068
 0.5       0.4993      0.5000        165.2      0.084
 0.6       0.6000      0.6000        198.8      0.095
 0.7       0.7000      0.7000        198.2      0.098
 0.8       0.8000      0.8000        165.3      0.094
 0.9       0.9006      0.9000        116.0      0.083
 1.0       1.0000      1.0000         84.4      0.069


In [4]:
barrier = locate_barrier(g, e)
well_o = fit_octahedral_well(g, e)
well_t = fit_tetrahedral_well(g, e)
print(f"transition state: g = {barrier.g_ts:.3f}, E = {barrier.e_ts_ev*1e3:.1f} meV (spline {barrier.e_ts_spline_ev*1e3:.1f} meV)")
print(f"k_O = {well_o.k_g_ev:.3f} eV/g^2, k_T = {well_t.k_g_ev:.3f} eV/g^2")
fig = plot_energy_profile(g, e, barrier, well_o, well_t, e_t_unconstrained_ev=(tetra.final_energy - octa.final_energy) * HARTREE_EV)
fig.tight_layout()

transition state: g = 0.650, E = 203.8 meV (spline 203.0 meV)
k_O = 0.968 eV/g^2, k_T = 7.307 eV/g^2


## 3. Attempt frequencies, rates and the diffusion constant

The curvatures along g become force constants through the path length |T - O| = sqrt(3) a / 4, and the real hydrogen mass gives the attempt frequencies. Rates and populations are combined in the two-sublattice master-equation expression with the Einstein factor 1/2.

In [5]:
L = path_length_ang(A_PD_ANG)
w_o = attempt_frequency(force_constant_si(well_o.k_g_ev, L), MASS_H_AMU)
w_t = attempt_frequency(force_constant_si(well_t.k_g_ev, L), MASS_H_AMU)
print(f"hbar omega: O {vibrational_quantum_mev(w_o):.1f} meV, T {vibrational_quantum_mev(w_t):.1f} meV")
model = SiteModel(de_t_minus_o_ev=float(e[-1]), ea_o_to_t_ev=barrier.e_ts_ev, ea_t_to_o_ev=barrier.e_ts_ev - e[-1], omega_o=w_o, omega_t=w_t)
temps = np.linspace(200, 1000, 161)
res = diffusion_constant(model, temps, hop_geometry(A_PD_ANG))
ea, d0 = arrhenius_fit(temps, res.d_m2_s)
d298 = float(np.interp(298, temps, res.d_m2_s))
print(f"D(298 K) = {d298:.2e} m^2/s, experiment {D_EXP_298K_M2_S:.2e} m^2/s, ratio {d298 / D_EXP_298K_M2_S:.1f}")
print(f"Arrhenius fit: E_a = {ea:.3f} eV, D0 = {d0:.2e} m^2/s; detailed balance ratio {res.detailed_balance_ratio[0]:.3f}")
fig = plot_arrhenius(res, ea, d0)
fig.tight_layout()

hbar omega: O 37.6 meV, T 103.3 meV
D(298 K) = 2.40e-10 m^2/s, experiment 3.25e-11 m^2/s, ratio 7.4
Arrhenius fit: E_a = 0.198 eV, D0 = 5.27e-07 m^2/s; detailed balance ratio 1.000


## 4. Sensitivity to the population model

Three conventions for the sublattice populations, evaluated at 298 K. They move D by less than a factor two; the barrier and the missing zero-point energy dominate the gap to experiment.

In [6]:
for mode in ("harmonic1d", "harmonic3d", "boltzmann"):
    r = diffusion_constant(model, [298.0], hop_geometry(A_PD_ANG), population_mode=mode)
    print(f"{mode:12s} D(298 K) = {r.d_m2_s[0]:.2e} m^2/s   P_tet = {r.p_tet[0]:.3f}   flux ratio = {r.detailed_balance_ratio[0]:.2f}")

harmonic1d   D(298 K) = 2.39e-10 m^2/s   P_tet = 0.026   flux ratio = 1.00
harmonic3d   D(298 K) = 1.39e-10 m^2/s   P_tet = 0.004   flux ratio = 7.55
boltzmann    D(298 K) = 4.29e-10 m^2/s   P_tet = 0.069   flux ratio = 0.36
